In [6]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import re
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

df = pd.read_csv("ocean.csv")

def extract_price_num(price_str):
    if pd.isna(price_str):
        return None
    price_str = price_str.lower().replace(',', '.').replace('triệu', '').replace('vnđ', '')
    numbers = re.findall(r"[\d.]+", price_str)
    if numbers:
        return float(numbers[0])
    return None

df["price_num"] = df["price"].apply(extract_price_num)

def extract_location_detail(title):
    patterns = [
        r"(Hải Âu\s*\d+)",
        r"(San Hô)",
        r"(Masteri)",
        r"(P\d+)",
        r"(HA\d+)",
        r"(S\d+\s*-\s*\d+)",
        r"(TMDV\s*HA\d+)",
        r"(Thuận An)",
        r"(Lý Thanh Tông)",
        r"(Trâu Quỳ)",
        r"(Ocean Park.*?)(?:,|$)",
    ]
    for pat in patterns:
        match = re.search(pat, str(title), re.IGNORECASE)
        if match:
            return match.group(1).strip()
    return None

df["location_detail"] = df["title"].apply(extract_location_detail)

df["location_full"] = df.apply(
    lambda row: f"{row['location_detail']}, {row['location']}" if pd.notna(row["location_detail"]) else row["location"],
    axis=1
)

geolocator = Nominatim(user_agent="geoapi")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

def get_coords(location):
    try:
        loc = geocode(location)
        if loc:
            return pd.Series([loc.latitude, loc.longitude])
    except:
        return pd.Series([None, None])
    return pd.Series([None, None])

df[["lat", "lon"]] = df["location_full"].apply(get_coords)

df.dropna(subset=["lat", "lon"], inplace=True)
geometry = [Point(xy) for xy in zip(df["lon"], df["lat"])]
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

gdf.to_file("shophouse_data.geojson", driver="GeoJSON")  




In [7]:
import folium
import geopandas as gpd
import pandas as pd

gdf = gpd.read_file("shophouse_data.geojson")

m = folium.Map(
    location=[gdf["lat"].mean(), gdf["lon"].mean()],
    zoom_start=14,
    tiles="CartoDB positron"
)


for _, row in gdf.iterrows():
    popup_html = f"""
    <div style="width: 360px; font-family: 'Arial', sans-serif; font-size: 14px;">
        <b>Địa chỉ:</b> {row['location_full']}<br>
        <b>Giá:</b> {row['price']}<br>
        <b>Diện tích:</b> {row['area']} m²<br>
        <b>Số toilet:</b> {row['toilet'] if pd.notna(row['toilet']) else 'Không rõ'}<br>
        <b>Mô tả:</b> {row['description'] if pd.notna(row['description']) else 'Không có mô tả'}<br><br>
        <a href="{row['url']}" target="_blank">🔗 Xem chi tiết</a>
    </div>
    """

    iframe = folium.IFrame(html=popup_html, width=360, height=200)
    popup = folium.Popup(iframe, max_width=500)
    folium.Marker(location=[row["lat"], row["lon"]], popup=popup).add_to(m)

m.save("map.html")


In [8]:
folium.LayerControl().add_to(m)
m.save("interactive_map.html")


In [9]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# Dữ liệu mẫu
data = [
    {
        "id": 42819911,
        "price": "10,5 triệu/tháng",
        "area": 70.0,
        "floor": 2,
        "address": "Hải Âu 3, Gia Lâm, Hà Nội",
        "description": "Shophouse 70m², 2PN, 2WC, 2 tầng, không nội thất, gần Vincom Ocean Park.",
        "lat": 20.991787,
        "lon": 105.955071
    },
    {
        "id": 42819912,
        "price": "12 triệu/tháng",
        "area": 75.0,
        "floor": 3,
        "address": "Thuận An, Trâu Quỳ Gia Lâm",
        "description": "Mặt phố kinh doanh đẹp, gần Vinschool và siêu thị.",
        "lat": 21.011311,
        "lon": 105.952158
    },
    {
        "id": 40118935,
        "price": "26 triệu/tháng",
        "area": 50.0,
        "floor": 1,
        "address": "Dường Lý Thanh Tông, Gia Lâm",
        "description": "2 hầm gửi xe, 2 mặt tiền, vị trí quảng cáo thu hút ánh nhìn.",
        "lat": 21.008305555555555,
        "lon": 105.95272222222223
    }
]

df = pd.DataFrame(data)
geometry = [Point(xy) for xy in zip(df["lon"], df["lat"])]
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")


In [10]:
import folium

# Khởi tạo bản đồ
m = folium.Map(location=[df["lat"].mean(), df["lon"].mean()], zoom_start=13)

# Thêm marker với popup tùy chỉnh
for _, row in gdf.iterrows():
    popup_html = f"""
    <b>Địa chỉ:</b> {row['address']}<br>
    <b>Giá:</b> {row['price']}<br>
    <b>Diện tích:</b> {row['area']} m²<br>
    <b>Số tầng:</b> {row['floor']}<br>
    <b>Mô tả:</b> {row['description']}<br>
    """
    folium.Marker(
        location=[row["lat"], row["lon"]],
        popup=folium.Popup(popup_html, max_width=300),
        icon=folium.Icon(color="blue", icon="home")
    ).add_to(m)

# Lưu bản đồ
m.save("shophouse_map.html")
